**Name:** Nada Mohamed Khatab  
**id:** 4231230  

---
### Given Sequences

| ID | Sequence |
|----|----------|
| Seq1 | ACAATG |
| Seq2 | TCAACTATC |
| Seq3 | ACACAGC |
| Seq4 | AGAATC |
| Seq5 | ACCGATC |
| Seq6 | GATCAC |


---
## Part 0 — Setup


In [ ]:
!apt-get install -y -q mafft
print('MAFFT installed.')
!pip install biopython -q
print('Biopython ready.')

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from Bio import SeqIO, AlignIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

print('All libraries imported successfully.')

All libraries imported successfully.


---
## Part A — Multiple Sequence Alignment (MSA)


### A1 — Write FASTA File


In [ ]:
sequences = {
    'Seq1': 'ACAATG',
    'Seq2': 'TCAACTATC',
    'Seq3': 'ACACAGC',
    'Seq4': 'AGAATC',
    'Seq5': 'ACCGATC',
    'Seq6': 'GATCAC',
}

records = [SeqRecord(Seq(seq), id=name, description='') for name, seq in sequences.items()]
SeqIO.write(records, 'input.fasta', 'fasta')

with open('input.fasta') as f:
    print(f.read())

>Seq1
ACAATG
>Seq2
TCAACTATC
>Seq3
ACACAGC
>Seq4
AGAATC
>Seq5
ACCGATC
>Seq6
GATCAC



### A2 — Run MAFFT


In [ ]:
!mafft --auto input.fasta > aligned.fasta 2>/dev/null

with open('aligned.fasta') as f:
    print(f.read())

>Seq1
---acaatg--
>Seq2
tcaactatc--
>Seq3
ac-acagc---
>Seq4
---agaatc--
>Seq5
a--ccgatc--
>Seq6
-----gatcac



### A3 — Parse and Display the MSA

Read the aligned sequences and display them as a matrix table.


In [ ]:
# Read the aligned FASTA file
alignment = AlignIO.read('aligned.fasta', 'fasta')

seq_names = [rec.id for rec in alignment]
seq_strings = [str(rec.seq).upper() for rec in alignment]
n_cols = alignment.get_alignment_length()

# Build a DataFrame: rows = sequences, columns = alignment positions
matrix = {str(i+1): list(seq_strings[j]) for j in range(len(seq_names)) for i in range(n_cols)}

# build column by column
col_dict = {str(i+1): [seq[i] for seq in seq_strings] for i in range(n_cols)}
df_matrix = pd.DataFrame(col_dict, index=seq_names)
df_matrix.index.name = 'Sequence'

print('=== MSA Alignment Matrix ===')
print(df_matrix.to_string())
df_matrix

=== MSA Alignment Matrix ===
          1  2  3  4  5  6  7  8  9 10 11
Sequence                                 
Seq1      -  -  -  A  C  A  A  T  G  -  -
Seq2      T  C  A  A  C  T  A  T  C  -  -
Seq3      A  C  -  A  C  A  G  C  -  -  -
Seq4      -  -  -  A  G  A  A  T  C  -  -
Seq5      A  -  -  C  C  G  A  T  C  -  -
Seq6      -  -  -  -  -  G  A  T  C  A  C


,1,2,3,4,5,6,7,8,9,10,11
Sequence,,,,,,,,,,,
Seq1,-,-,-,A,C,A,A,T,G,-,-
Seq2,T,C,A,A,C,T,A,T,C,-,-
Seq3,A,C,-,A,C,A,G,C,-,-,-
Seq4,-,-,-,A,G,A,A,T,C,-,-
Seq5,A,-,-,C,C,G,A,T,C,-,-
Seq6,-,-,-,-,-,G,A,T,C,A,C


### A4 — Gap Fraction Analysis

In [ ]:
THRESHOLD = 0.35
n_seqs = len(seq_strings)
rows = []

for i in range(n_cols):
    residues = [seq[i] for seq in seq_strings]
    gap_count = residues.count('-')
    gap_frac = gap_count / n_seqs
    classification = 'M' if gap_frac <= THRESHOLD else 'I'
    rows.append({
        'Column': i + 1,
        'Residues': ', '.join(residues),
        'Gap Count': gap_count,
        'Gap Fraction': round(gap_frac, 2),
        'Classification': classification
    })

df_gap = pd.DataFrame(rows)
print('=== Gap Fraction Analysis ===')
df_gap

=== Gap Fraction Analysis ===


,Column,Residues,Gap Count,Gap Fraction,Classification
0,1,"-, T, A, -, A, -",3,0.50,I
1,2,"-, C, C, -, -, -",4,0.67,I
2,3,"-, A, -, -, -, -",5,0.83,I
3,4,"A, A, A, A, C, -",1,0.17,M
4,5,"C, C, C, G, C, -",1,0.17,M
5,6,"A, T, A, A, G, G",0,0.00,M
6,7,"A, A, G, A, A, A",0,0.00,M
7,8,"T, T, C, T, T, T",0,0.00,M
8,9,"G, C, -, C, C, C",1,0.17,M
9,10,"-, -, -, -, -, A",5,0.83,I


In [ ]:
# Summary of seed vs non-seed columns
m_cols = df_gap[df_gap['Classification'] == 'M']['Column'].tolist()
i_cols = df_gap[df_gap['Classification'] == 'I']['Column'].tolist()

print(f'Match (seed) columns    [M]: {m_cols}')
print(f'Insert (non-seed) columns [I]: {i_cols}')

Match (seed) columns    [M]: [4, 5, 6, 7, 8, 9]
Insert (non-seed) columns [I]: [1, 2, 3, 10, 11]


---
## Part B — Profile HMM Construction


### B1 — Identify Seed Columns

In [ ]:
col_classes = df_gap['Classification'].tolist()

print('Column classifications (1-indexed):')
for i, cls in enumerate(col_classes):
    print(f'  Column {i+1}: {cls}')

Column classifications (1-indexed):
  Column 1: I
  Column 2: I
  Column 3: I
  Column 4: M
  Column 5: M
  Column 6: M
  Column 7: M
  Column 8: M
  Column 9: M
  Column 10: I
  Column 11: I


### B2 — Build State Paths

In [ ]:
def build_state_path(aligned_seq, col_classes):
    """Build the HMM state path for one aligned sequence."""
    path = ['S']
    for i, char in enumerate(aligned_seq):
        cls = col_classes[i]
        if cls == 'M':
            path.append('M' if char != '-' else 'D')
        else:
            if char != '-':
                path.append('I')
    path.append('E')
    return path


# Build paths for all 6 sequences
path_rows = []
for name, seq in zip(seq_names, seq_strings):
    path = build_state_path(seq, col_classes)
    path_str = ' → '.join(path)
    path_rows.append({'Sequence': name, 'State Path': path_str})

df_paths = pd.DataFrame(path_rows)
print('=== Profile HMM State Paths ===')
df_paths

=== Profile HMM State Paths ===


,Sequence,State Path
0,Seq1,S → M → M → M → M → M → M → E
1,Seq2,S → I → I → I → M → M → M → M → M → M → E
2,Seq3,S → I → I → M → M → M → M → M → D → E
3,Seq4,S → M → M → M → M → M → M → E
4,Seq5,S → I → M → M → M → M → M → M → E
5,Seq6,S → D → D → M → M → M → M → I → I → E


In [ ]:
# Pretty-print the paths
print('\nProfile HMM State Paths\n' + '='*60)
for _, row in df_paths.iterrows():
    print(f"{row['Sequence']}: {row['State Path']}")


Profile HMM State Paths
Seq1: S → M → M → M → M → M → M → E
Seq2: S → I → I → I → M → M → M → M → M → M → E
Seq3: S → I → I → M → M → M → M → M → D → E
Seq4: S → M → M → M → M → M → M → E
Seq5: S → I → M → M → M → M → M → M → E
Seq6: S → D → D → M → M → M → M → I → I → E


### Summary of Steps

1. **Input:** 6 raw DNA sequences written to `input.fasta`
2. **MAFFT:** Aligned sequences saved to `aligned.fasta` — gaps inserted to equalize lengths
3. **Gap Fraction:** Each column scored; columns with ≤ 35% gaps = Match (seed); >35% = Insert (non-seed)
4. **Profile HMM:** Each sequence converted to a state path (S, M, D, I, E)


In [ ]:
# Download Output Files
from google.colab import files

# Save alignment matrix and state paths to CSV
df_matrix.to_csv('msa_matrix.csv')
df_gap.to_csv('gap_analysis.csv', index=False)
df_paths.to_csv('state_paths.csv', index=False)

# Download everything
files.download('aligned.fasta')
files.download('msa_matrix.csv')
files.download('gap_analysis.csv')
files.download('state_paths.csv')